# Part 2: Simple Logistic Regression Model
We will be using a simple bag-of-words representation for our initial, simple model. To train this model, we will be using the pytorch library.

In [ ]:
from transformers import *
from pathlib import Path
import numpy as np

DATA_DIR = Path("data/preprocessed")
TOKENIZER_DIR = DATA_DIR / "tokenizers"
SPLIT_DIR = DATA_DIR / "splits"

In [ ]:
from sklearn.pipeline import Pipeline
from joblib import load


tokenizer : TokenTransformer = load(TOKENIZER_DIR / "top10k.joblib")
pipeline = Pipeline([
    ("tokenize", PrefitTransformer(tokenizer)), 
    ("vectorize", BagofwordsTransformer("content", tokenizer.size()))
])
pipeline

In [ ]:
import pandas as pd
df_train = pd.read_csv(SPLIT_DIR / "train.csv")
df_train.head()

In [ ]:
df_val = pd.read_csv(SPLIT_DIR / "val.csv")
df_val.head()

In [ ]:
X_train, y_train = pipeline.fit_transform(df_train), df_train["is_reliable"].to_numpy()
X_val, y_val = pipeline.transform(df_val), df_val["is_reliable"].to_numpy()

In [ ]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    verbose=1)

print("fitting")
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import f1_score

y_pred = model.predict(X_val)
print("F1 score of logistic regression model: ", f1_score(y_val, y_pred))

## 2.1. Including content
As we saw in the data analysis section, the label of an article is strongly controlled by the domain it originates from, so including this information should surely improve the model

In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


tokenizer : TokenTransformer = load(TOKENIZER_DIR / "top10k.joblib")
pipeline = Pipeline([
    ("transform", ColumnTransformer(
        transformers=[
            ("domain", OneHotEncoder(handle_unknown="ignore"), ["domain"]),
            ("content", Pipeline([
                ("tokenize", PrefitTransformer(tokenizer)), 
                ("vectorize", BagofwordsTransformer("content", tokenizer.size()))
            ]), ["content"])
        ]))
])
pipeline

In [ ]:
X_train, y_train = pipeline.fit_transform(df_train), df_train["is_reliable"].to_numpy()
X_val, y_val = pipeline.transform(df_val), df_val["is_reliable"].to_numpy()

In [ ]:
model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    verbose=1)

print("fitting")
model.fit(X_train,y_train)

In [ ]:
from sklearn.metrics import f1_score

y_pred = model.predict(X_val)
print("F1 score of logistic regression model including domain: ", f1_score(y_val, y_pred))